# End-to-end analysis
Create a synthetic CSV, load it, run independent tests and inspect input row positions. An anomaly is not proof of wrongdoing; check each result's caveats before interpretation.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tempfile import TemporaryDirectory
from fraudscope.ingest import load
from fraudscope.runner import Config, RegressionSpec, run_all
from fraudscope.plots import plot_summary_dashboard
rng = np.random.default_rng(14)
raw = pd.DataFrame({'amount': 10**rng.uniform(1, 6, 1000), 'sequence': np.arange(1000)})
with TemporaryDirectory() as directory:
    path = Path(directory) / 'sample.csv'
    raw.to_csv(path, index=False)
    df = load(path)
    results = run_all(df, Config(columns=['amount'], benford_modes=['first', 'second'], references={'amount': 'benford'}, regression=RegressionSpec('amount', ['sequence']), random_state=14))
    print(df.attrs['cleaning'])
    print([(r.test_name, r.p_value, r.caveats) for r in results])
    plot_summary_dashboard(results)
    print(df.iloc[results[0].flagged_indices[:5]])